# 01 — The evaluation set and the metrics harness

**Group 4 — Delta Air Lines Customer Support Assistant**

The methodology is explicit that the evaluation set must exist *before* any design
decision is made. This notebook builds and validates the ruler; notebooks 02-04 use it.

## The labelling problem, and why it dictates everything else

Stage 1 varies the PDF parser and Stage 2 varies the chunking strategy. Both change
chunk boundaries and therefore chunk IDs. So ground truth **cannot** be recorded as
"chunk #47 is correct" — those labels would be silently invalidated by the first two
ablations, and every table downstream would be measuring nothing.

Instead labels are recorded at a level that survives both:

| Field | Meaning |
|---|---|
| `gold_doc` | stable document id |
| `gold_locators` | PDF page as `p17`, or XBRL concept as `us-gaap:NetIncomeLoss` |
| `gold_span` | verbatim answer-bearing text |

A page number is agreed by every parser; an XBRL concept is agreed by every chunker.

In [ ]:
import sys; sys.path.insert(0, "../src")
import pandas as pd
from delta_rag.evalset import load_eval_set, summarise, validate_against_corpus, span_recovery
from delta_rag.corpus import load_corpus

questions = load_eval_set()
summarise(questions)

30 questions — above the 20+ floor. The `query_type` split is not decoration: Stage 5
requires R@3 broken out by keyword vs semantic queries, so that label has to be
authored up front rather than reconstructed later.

Two questions are deliberate **routing hard-negatives**: `F03` ("How much revenue did
Delta make from shipping cargo last year?") is saturated with cargo vocabulary but the
answer is a financial fact, and `R2` in the acceptance set shares "liability" with the
passenger contract while asking about cargo.

In [ ]:
pd.DataFrame([
    {"id": q.id, "route": q.route, "type": q.query_type,
     "locators": ",".join(q.gold_locators)[:34], "question": q.question[:64]}
    for q in questions
])

## Validating the labels

An unverified label set is worse than none, because it produces confident wrong
numbers. Every `gold_span` is checked to occur verbatim at its `gold_locator`.

In [ ]:
problems = validate_against_corpus(questions, load_corpus("pymupdf"))
print(f"label problems: {len(problems)}")
for p in problems:
    print(" ", p)

pd.DataFrame([
    {"parser": p, "gold spans recoverable": f"{span_recovery(questions, load_corpus(p)):.1%}"}
    for p in ("pymupdf", "pypdf", "pdfplumber")
])

This validator earned its place twice.

1. It caught a **real labelling error** of mine: EPS renders as `USD 7.66` because the
   XBRL unit is `usdPerShare`, and my label omitted `USD`.
2. It caught a **real parser defect**: pypdf extracts `those` as `t hose` on contract
   page 4, so one gold span is unrecoverable from its output.

The second is Stage 1 evidence, not a bad label — which forced an explicit decision
about how much that defect should cost.

## Graded relevance, and the binary threshold

| Grade | Meaning |
|---|---|
| 3 | chunk contains the verbatim gold span (answer-bearing) |
| 2 | chunk is in the gold doc at a gold locator (right place, span degraded) |
| 1 | chunk is in the gold doc elsewhere |
| 0 | wrong document |

R@1/R@3/MRR@10 count relevance at **grade ≥ 2**, not grade 3. Requiring the verbatim
span would score pypdf's mangled-but-correct page as a *total retrieval miss*, which
overstates the damage and double-counts it — the page is still the right page and still
answers the question. Instead the defect costs NDCG@3 (graded) while leaving R@3
(binary) intact, so parser damage is counted once, in one place.

In [ ]:
import math
from delta_rag.metrics import dcg_at_k, ndcg_at_k, recall_at_k, reciprocal_rank

# Hand-worked check: DCG@3 = 3/log2(2) + 2/log2(3) + 1/log2(4)
expected = 3 / math.log2(2) + 2 / math.log2(3) + 1 / math.log2(4)
print(f"DCG@3([3,2,1]) = {dcg_at_k([3,2,1],3):.5f}  (expected {expected:.5f})")
print(f"NDCG@3 perfect order   = {ndcg_at_k([3,2,1],3):.4f}")
print(f"NDCG@3 worst order     = {ndcg_at_k([1,2,3],3):.4f}   <- same grades, worse ranking")
print(f"R@1([1,0,0])           = {recall_at_k([1,0,0],1)}   <- grade 1 is below threshold")
print(f"R@1([2,0,0])           = {recall_at_k([2,0,0],1)}   <- grade 2 is at threshold")
print(f"MRR([0,0,2])           = {reciprocal_rank([0,0,2],10):.4f}")

The full harness is pinned by 30 unit tests (`pytest tests/test_metrics.py`), including
a test that asserts the relevance threshold is 2 so the decision above cannot drift
silently.